# Template Quest Notebook

**Hypothesis:** An adversary delivers a spearphishing attachment. The Office
application that opens the document makes an outbound web call to
attacker-controlled infrastructure. Sysmon Event ID 22 (DnsQuery) on the
Windows endpoint is the observable.


## Part 1: Load the Sample Dataset

### Environment setup (Google Colab only)

Skip these two cells if you are running inside the cloned repository with
PySpark already installed.

Colab ships `dataproc-spark-connect`, which patches `SparkSession.builder` into
Spark Connect mode and prevents a local session from starting. It has to be
removed, and the runtime restarted, before Spark will initialise locally.

In [1]:
# Colab setup - uninstall Spark Connect, install PySpark
!pip uninstall -y pyspark pyspark-connect dataproc-spark-connect google-spark-connect
!pip cache purge
!pip install --force-reinstall --no-cache-dir pyspark==3.5.1

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Clear any Spark Connect variables. If SPARK_REMOTE is set, or connect mode is
# enabled, SparkSession.builder tries to reach a remote cluster instead of
# starting a local JVM, and fails with CONNECT_URL_NOT_SET.
os.environ.pop("SPARK_REMOTE", None)
os.environ.pop("SPARK_CONNECT_MODE_ENABLED", None)

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 223.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 183.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=32fd36ee9469be227411cb311d965e11a7293f3764607da39dd5a3a282261368
  Stored in directory: /tmp/pip-ephem-wheel-cache-dx9k0e3v/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7


In [2]:
# Colab setup - fetch the repo and move into it so relative paths resolve.
!git clone -q https://github.com/rearc/cyber-quest.git
%cd cyber-quest
!ls -lh data/

fatal: destination path 'cyber-quest' already exists and is not an empty directory.
/content/cyber-quest
total 40M
-rw-r--r-- 1 root root 40M Aug  2 16:08 sysmon_spearphish_cribl.json


In [3]:
# Import neccessary modules


from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StringType
import requests
import pyspark, os

print(pyspark.__version__)
print(pyspark.__file__)

3.5.1
/usr/local/lib/python3.12/dist-packages/pyspark/__init__.py


In [4]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

In [6]:
# Load the raw data
df_bronze = spark.read.json("data/sysmon_spearphish_cribl.json")

print(f"Loaded {df_bronze.count()} events")
df_bronze.printSchema()

# --- pandas equivalent -------------------------------------------------------
# import pandas as pd
# df_bronze = pd.read_json("data/sysmon_spearphish_cribl.json", lines=True)
# print(len(df_bronze), df_bronze.columns.tolist())
#
# -----------------------------------------------------------------------------

Loaded 28017 events
root
 |-- Computer: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- EventCode: string (nullable = true)
 |-- User: string (nullable = true)
 |-- _raw: string (nullable = true)
 |-- _time: double (nullable = true)
 |-- cribl_breaker: string (nullable = true)
 |-- cribl_pipe: string (nullable = true)
 |-- host: string (nullable = true)
 |-- source: string (nullable = true)



### Inspecting the raw payload before parsing

Cribl promotes a handful of fields to the top level (`Computer`, `EventCode`,
`User`, `Description`, `host`, `source`, `_time`) and leaves the full Sysmon
event as a string in `_raw`.

Cribl can ship Windows event data as nested JSON, XML, or key=value depending on
pipeline configuration, and the parse strategy differs for each. So I inspect
`_raw` before writing any parsing logic rather than assuming a format.

In [7]:
df_bronze.select("_raw").show(1, truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 _raw | {"Name":"'Microsoft-Windows-Sysmon'","Guid":"'{5770385F-C22A-43E0-BF4C-06F5698

The payload is **flat JSON** with all Sysmon fields at the top level, so
`from_json` with an explicit schema is the right tool.

I declare the schema explicitly rather than relying on inference. Inference
samples the data to guess types, which means the parse can silently change shape
if the input changes. For a detection pipeline that needs to behave the same way
every run, an explicit contract is worth the extra few lines.

In [8]:
# Parse the raw data into something relevant and usable
sysmon_schema = (StructType()
    .add("UtcTime", StringType())
    .add("SystemTime", StringType())
    .add("EventRecordID", StringType())
    .add("ProcessGuid", StringType())
    .add("ProcessId", StringType())
    .add("Image", StringType())
    .add("QueryName", StringType())
    .add("QueryStatus", StringType())
    .add("QueryResults", StringType())
    .add("UserID", StringType())
    .add("RuleName", StringType())
    .add("Channel", StringType()))

df_silver = (df_bronze
    .withColumn("event", F.from_json(F.col("_raw"), sysmon_schema))
    .select("Computer", "User", "Description", "host", "EventCode", "event.*")
    .withColumn("event_time", F.to_timestamp(F.col("UtcTime")))
    .withColumn("image_lc", F.lower(F.col("Image"))))

df_silver.createOrReplaceTempView("sysmon_silver")

print(f"Parsed {df_silver.count()} events")

# --- pandas equivalent -------------------------------------------------------
# import json
# parsed = pd.json_normalize(df_bronze["_raw"].apply(json.loads))
# df_silver = pd.concat(
#     [df_bronze[["Computer", "User", "Description", "host", "EventCode"]]
#         .reset_index(drop=True),
#      parsed.reset_index(drop=True)], axis=1)
# df_silver["event_time"] = pd.to_datetime(df_silver["UtcTime"])
# df_silver["image_lc"] = df_silver["Image"].str.lower()
#
# json_normalize flattens the parsed dicts into columns so downstream filtering
# stays vectorised, rather than reaching into a dict per row.
# -----------------------------------------------------------------------------

Parsed 28017 events


In [9]:
# Provided PySpark Example
df_silver.limit(5).show()

+--------------------+-------------------+-----------+------------+---------+--------------------+--------------------+-------------+--------------------+---------+--------------------+---------+-----------+------------+--------+--------+--------------------+--------------------+--------------------+
|            Computer|               User|Description|        host|EventCode|             UtcTime|          SystemTime|EventRecordID|         ProcessGuid|ProcessId|               Image|QueryName|QueryStatus|QueryResults|  UserID|RuleName|             Channel|          event_time|            image_lc|
+--------------------+-------------------+-----------+------------+---------+--------------------+--------------------+-------------+--------------------+---------+--------------------+---------+-----------+------------+--------+--------+--------------------+--------------------+--------------------+
|win-host-ctus-att...|NT AUTHORITY\SYSTEM|       NULL|f2c75b47cbb7|       23|2023-01-27 11:19:

In [10]:
# Provided PySpark SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
LIMIT 5
""").show()

+--------------------+-------------------+-----------+------------+---------+--------------------+--------------------+-------------+--------------------+---------+--------------------+---------+-----------+------------+--------+--------+--------------------+--------------------+--------------------+
|            Computer|               User|Description|        host|EventCode|             UtcTime|          SystemTime|EventRecordID|         ProcessGuid|ProcessId|               Image|QueryName|QueryStatus|QueryResults|  UserID|RuleName|             Channel|          event_time|            image_lc|
+--------------------+-------------------+-----------+------------+---------+--------------------+--------------------+-------------+--------------------+---------+--------------------+---------+-----------+------------+--------+--------+--------------------+--------------------+--------------------+
|win-host-ctus-att...|NT AUTHORITY\SYSTEM|       NULL|f2c75b47cbb7|       23|2023-01-27 11:19:

## Part 2: Detection Engineering

### Profiling before filtering

Before writing a rule I want to know what telemetry actually exists, and what
normal looks like. Writing the detection first and checking the data afterwards
is how allowlists end up tuned to assumptions instead of evidence.

In [11]:
# What kinds of events are in this dataset?
df_silver.groupBy("EventCode").count().orderBy(F.desc("count")).show()

# --- pandas equivalent -------------------------------------------------------
# df_silver["EventCode"].value_counts()
# -----------------------------------------------------------------------------

+---------+-----+
|EventCode|count|
+---------+-----+
|       23|11483|
|       10| 9673|
|       13| 4125|
|       11| 1790|
|        3|  648|
|        1|  248|
|       18|   17|
|       22|   16|
|       17|    7|
|        2|    5|
|        7|    4|
|       12|    1|
+---------+-----+



The dataset is dominated by file deletes (EventCode 23) and process access
(EventCode 10). DNS queries (EventCode 22) are a small fraction.

That count is low enough to be worth verifying rather than assuming a filter
bug - and it is correct. It also means the DNS subset is small enough to review
by hand, which I do below.

In [12]:
# Which processes query which domains? This is the evidence the allowlist rests on.
(df_silver
    .filter(F.col("EventCode") == "22")
    .groupBy("image_lc", "QueryName")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False))

# --- pandas equivalent -------------------------------------------------------
# dns = df_silver[df_silver["EventCode"].astype(str) == "22"]
# dns.groupby(["image_lc", "QueryName"]).size().sort_values(ascending=False)
# -----------------------------------------------------------------------------

+------------------------------------------------------------------------------+---------------------------------+-----+
|image_lc                                                                      |QueryName                        |count|
+------------------------------------------------------------------------------+---------------------------------+-----+
|c:\program files\common files\microsoft shared\clicktorun\officeclicktorun.exe|ecs.office.com                   |2    |
|c:\program files\common files\microsoft shared\clicktorun\officeclicktorun.exe|f.c2r.ts.cdn.office.net          |2    |
|c:\windows\system32\svchost.exe                                               |f.c2r.ts.cdn.office.net          |1    |
|c:\temp\officesetup.exe                                                       |ecs.office.com                   |1    |
|c:\windows\system32\svchost.exe                                               |oneclient.sfx.ms                 |1    |
|c:\program files\microsoft offi

### Reviewing every DNS event

With only a handful of DNS events, reviewing all of them individually is both
feasible and more informative than any aggregate.

In [13]:
(df_silver
    .filter(F.col("EventCode") == "22")
    .select("event_time", "Computer", "Image", "QueryName", "QueryStatus")
    .orderBy("event_time")
    .show(50, truncate=False))

+-----------------------+------------------------------+------------------------------------------------------------------------------+---------------------------------+-----------+
|event_time             |Computer                      |Image                                                                         |QueryName                        |QueryStatus|
+-----------------------+------------------------------+------------------------------------------------------------------------------+---------------------------------+-----------+
|2023-01-27 11:22:32.315|win-host-ctus-attack-range-212|C:\Temp\OfficeSetup.exe                                                       |ecs.office.com                   |0          |
|2023-01-27 11:22:42.219|win-host-ctus-attack-range-212|C:\Program Files\Common Files\Microsoft Shared\ClickToRun\OfficeClickToRun.exe|ecs.office.com                   |0          |
|2023-01-27 11:22:42.694|win-host-ctus-attack-range-212|C:\Program Files\Common Files\Micr

### Detection logic

Two conditions:

1. **The querying process is an Office application.** Word's job is to render
   documents; it has no inherent reason to resolve arbitrary hostnames. When it
   does, something inside the document is driving it.

2. **The queried domain is not Microsoft telemetry.** Every legitimate Office
   query in this data resolved under `office.com` or `office.net` - update
   checks, template CDN, telemetry endpoints.

**Two deliberate choices, both defensible under challenge:**

*Basename comparison, not path-tail matching.* I split the image path on the
separator and compare the final element against an explicit set. A naive
`endswith("winword.exe")` would also match a file named `notwinword.exe`, which
is a trivially easy rule to evade.

*Suffix allowlist, not an enumerated hostname list.* Microsoft adds and retires
CDN hostnames frequently. A list of specific hosts goes stale and starts
producing false positives without anyone noticing. A suffix rule survives that
churn with no ongoing maintenance.

The trade-off I am accepting: an attacker who controlled a subdomain under an
allowlisted suffix would be suppressed. That is low-likelihood and high-impact,
and I note the mitigation in the limitations section.

In [14]:
OFFICE_PROCS = ["winword.exe", "excel.exe", "powerpnt.exe",
                "msaccess.exe", "outlook.exe"]

MS_TELEMETRY_SUFFIXES = [".office.com", ".office.net"]

# Condition 1: extract the basename from the path and compare against the set.
is_office_process = (
    F.element_at(F.split(F.col("image_lc"), "\\\\"), -1).isin(OFFICE_PROCS))

# Condition 2: suffix match, so new Microsoft CDN hostnames stay suppressed.
is_ms_telemetry = F.lit(False)
for suffix in MS_TELEMETRY_SUFFIXES:
    is_ms_telemetry = is_ms_telemetry | F.col("QueryName").endswith(suffix)

office_dns_hits = (df_silver
    .filter(F.col("EventCode") == "22")
    .filter(is_office_process)
    .filter(~is_ms_telemetry)
    .select("event_time", "Computer", "User", "Image", "ProcessGuid",
            "ProcessId", "QueryName", "QueryStatus", "QueryResults"))

office_dns_hits.createOrReplaceTempView("office_dns_hits")
office_dns_hits.show(truncate=False)

# --- pandas equivalent -------------------------------------------------------
# office_dns_hits = df_silver[
#     (df_silver["EventCode"].astype(str) == "22")
#     & df_silver["image_lc"].str.split("\\").str[-1].isin(OFFICE_PROCS)
#     & ~df_silver["QueryName"].str.endswith(tuple(MS_TELEMETRY_SUFFIXES))
# ][["event_time", "Computer", "User", "Image", "ProcessGuid",
#    "ProcessId", "QueryName", "QueryStatus", "QueryResults"]]
#
# .str.endswith() accepts a tuple of suffixes directly, so the allowlist check
# collapses to a single call with no regex and no escaping.
# -----------------------------------------------------------------------------

+-----------------------+------------------------------+----+-----------------------------------------------------------+--------------------------------------+---------+-----------------+-----------+----------------------------------------+
|event_time             |Computer                      |User|Image                                                      |ProcessGuid                           |ProcessId|QueryName        |QueryStatus|QueryResults                            |
+-----------------------+------------------------------+----+-----------------------------------------------------------+--------------------------------------+---------+-----------------+-----------+----------------------------------------+
|2023-01-27 11:30:14.523|win-host-ctus-attack-range-212|NULL|C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|{72106695-B5C3-63D3-3D04-00000000BD02}|4200     |www.mediafire.com|0          |::ffff:104.16.54.48;::ffff:104.16.53.48;|
+-----------------------+-------

### Equivalent detection in SQL and SPL

The same logic expressed as Spark SQL:

```sql
SELECT event_time, Computer, User, Image, ProcessGuid, ProcessId,
       QueryName, QueryStatus, QueryResults
FROM sysmon_silver
WHERE EventCode = '22'
  AND (image_lc LIKE '%winword.exe'  OR image_lc LIKE '%excel.exe'
    OR image_lc LIKE '%powerpnt.exe' OR image_lc LIKE '%msaccess.exe'
    OR image_lc LIKE '%outlook.exe')
  AND QueryName NOT LIKE '%.office.com'
  AND QueryName NOT LIKE '%.office.net'
```

And as SPL (If Splunk is being used) -

```
index=<sysmon_index> sourcetype="XmlWinEventLog:Microsoft-Windows-Sysmon/Operational"
  EventCode=22
| rex field=Image "(?<process_name>[^\\\\]+)$"
| eval process_name=lower(process_name)
| search process_name IN ("winword.exe","excel.exe","powerpnt.exe",
                          "msaccess.exe","outlook.exe")
| where NOT match(QueryName, "(?i)\.office\.(com|net)$")
```

The SPL version uses `rex` for the basename extraction, which makes it slightly
stricter than the SQL `LIKE` form.

### Result

One event survives the filter.

**Microsoft Word resolved `www.mediafire.com`.**

Word has no legitimate reason to resolve a consumer file-sharing host. Read
against the surrounding events - every other Word query in the same session went
to Microsoft telemetry - this is consistent with macro or embedded-object
execution inside a delivered document reaching out to stage a second-stage
payload.

### Observations outside the stated hypothesis

Two further artifacts that I would raise in a real triage, though they fall
outside the DNS-from-Office hypothesis:

- **`C:\Temp\OfficeSetup.exe`** made a DNS query several minutes earlier. A
  genuine Office installer does not run from `C:\Temp`. This is a binary
  masquerading as Microsoft tooling.
- **`<unknown process>`** made a query shortly after. Sysmon could not resolve
  the image name, which typically means the process had already exited by the
  time the event was written - consistent with a short-lived dropper or
  injected thread.

All activity sits on a single host inside a nine-minute window. That temporal
and host clustering is itself corroborating signal.

In [19]:
# Timeline of the surrounding activity, for context around the detection.
(df_silver
    .filter(F.col("EventCode") == "22")
    .select("event_time", "Image", "QueryName")
    .orderBy("event_time")
    .show(50, truncate=False))

+-----------------------+------------------------------------------------------------------------------+---------------------------------+
|event_time             |Image                                                                         |QueryName                        |
+-----------------------+------------------------------------------------------------------------------+---------------------------------+
|2023-01-27 11:22:32.315|C:\Temp\OfficeSetup.exe                                                       |ecs.office.com                   |
|2023-01-27 11:22:42.219|C:\Program Files\Common Files\Microsoft Shared\ClickToRun\OfficeClickToRun.exe|ecs.office.com                   |
|2023-01-27 11:22:42.694|C:\Program Files\Common Files\Microsoft Shared\ClickToRun\OfficeClickToRun.exe|ecs.office.com                   |
|2023-01-27 11:22:42.766|C:\Program Files\Common Files\Microsoft Shared\ClickToRun\OfficeClickToRun.exe|f.c2r.ts.cdn.office.net          |
|2023-01-27 11:22:42.83 |C:

## Part 3: Additional Steps

### Part 3.1: Normalization

I map the result to **OCSF DNS Activity, `class_uid` 4003**.

The reason for normalising at all: a DNS event from Sysmon, from a firewall, and
from a resolver all describe the same real-world thing but name their fields
differently. Downstream consumers - correlation rules, dashboards, the analyst
reading the alert - should not need to know or care which sensor produced the
record. Normalising is what makes that possible.

I chose OCSF over ECS mainly for its explicit DNS Activity class and its
process/device/actor structure, which maps cleanly onto what Sysmon gives us.

| OCSF field | Source |
|---|---|
| `class_uid` | 4003 (constant) |
| `activity_id` | 1 - Query (constant) |
| `time` | `UtcTime` |
| `process.file_name` | `Image` |
| `process.pid` | `ProcessId` |
| `process.uid` | `ProcessGuid` |
| `device.hostname` | `Computer` |
| `actor_user.name` | `User` |
| `query.hostname` | `QueryName` |
| `answers` | `QueryResults` |
| `metadata.product` | Sysmon (constant) |

In [20]:
ocsf_events = office_dns_hits.select(
    F.lit(4003).alias("class_uid"),
    F.lit("DNS Activity").alias("class_name"),
    F.lit(1).alias("activity_id"),          # 1 = Query
    F.col("event_time").alias("time"),

    F.struct(
        F.col("Image").alias("file_name"),
        F.col("ProcessId").cast("int").alias("pid"),
        F.col("ProcessGuid").alias("uid")
    ).alias("process"),

    F.struct(F.col("Computer").alias("hostname")).alias("device"),
    F.struct(F.col("User").alias("name")).alias("actor_user"),
    F.struct(F.col("QueryName").alias("hostname")).alias("query"),

    F.col("QueryResults").alias("answers"),

    F.struct(
        F.lit("Sysmon").alias("product"),
        F.lit("Cribl").alias("pipeline")
    ).alias("metadata"))

# printSchema demonstrates the structural change, not just different labels.
ocsf_events.printSchema()
ocsf_events.show(truncate=False, vertical=True)

# --- pandas note -------------------------------------------------------------
# pandas has no native struct type, so the OCSF nesting would be built as dicts
# per row and serialised to JSON, losing the schema enforcement Spark gives here.
# This is one place where the Spark implementation is genuinely better suited.
# -----------------------------------------------------------------------------

root
 |-- class_uid: integer (nullable = false)
 |-- class_name: string (nullable = false)
 |-- activity_id: integer (nullable = false)
 |-- time: timestamp (nullable = true)
 |-- process: struct (nullable = false)
 |    |-- file_name: string (nullable = true)
 |    |-- pid: integer (nullable = true)
 |    |-- uid: string (nullable = true)
 |-- device: struct (nullable = false)
 |    |-- hostname: string (nullable = true)
 |-- actor_user: struct (nullable = false)
 |    |-- name: string (nullable = true)
 |-- query: struct (nullable = false)
 |    |-- hostname: string (nullable = true)
 |-- answers: string (nullable = true)
 |-- metadata: struct (nullable = false)
 |    |-- product: string (nullable = false)
 |    |-- pipeline: string (nullable = false)

-RECORD 0------------------------------------------------------------------------------------------------------------------
 class_uid   | 4003                                                                                            

### Part 3.2: Alert Table

The question I asked when choosing fields: *what does an analyst need before
they open the raw event?*

- **Identity and verdict** - rule name, severity, status
- **Where to go** - host, process, timestamp
- **What to look for next** - MITRE technique, so the analyst knows the family
  of attack and its likely follow-on behaviour
- **Pivot handles** - `ProcessGuid` is stable across Sysmon event types, so it
  is the key that links this DNS event to process creation, file writes, and
  network connections from the same process
- **A readable one-liner** - so the alert is triageable from a dashboard tile
  without clicking through

**On `alert_id`:** it is a deterministic hash of host + process GUID + domain +
timestamp, not a random UUID. If the detection re-runs over an overlapping time
window - which any scheduled job with lag tolerance will do - a random id
produces duplicate alerts in the queue. A deterministic one produces the same
id, and downstream deduplication becomes trivial.

**MITRE mapping:** T1566.001 (Spearphishing Attachment) as the primary
technique, matching the hypothesis. T1105 (Ingress Tool Transfer) referenced in
enrichment as the follow-on behaviour - reaching out to a file host to pull a
payload - which is what an analyst should hunt for next.

In [21]:
alert_records = office_dns_hits.select(

    # Deterministic id: same event always produces the same alert_id.
    F.sha2(
        F.concat_ws("|",
                    F.col("Computer"),
                    F.col("ProcessGuid"),
                    F.col("QueryName"),
                    F.col("event_time").cast("string")),
        256
    ).substr(1, 16).alias("alert_id"),

    F.col("event_time").alias("detected_at"),

    F.lit("Office Application DNS Query to Non-Microsoft Domain").alias("rule_name"),
    F.lit("1.0").alias("rule_version"),
    F.lit("T1566.001").alias("mitre_technique"),
    F.lit("Spearphishing Attachment").alias("mitre_name"),
    F.lit("High").alias("severity"),
    F.lit("new").alias("status"),

    F.col("Computer").alias("host"),
    F.col("User").alias("user"),
    F.col("Image").alias("process"),
    F.col("ProcessGuid").alias("process_guid"),

    F.col("QueryName").alias("indicator"),
    F.lit("dns_domain").alias("indicator_type"),

    F.concat(
        F.lit("Office application resolved "),
        F.col("QueryName"),
        F.lit(" - outside allowed Microsoft telemetry domains")
    ).alias("summary"))

alert_records.show(truncate=False, vertical=True)

# --- pandas equivalent -------------------------------------------------------
# import hashlib
# alert_records = office_dns_hits.assign(
#     alert_id=lambda d: d.apply(lambda r: hashlib.sha256(
#         f"{r.Computer}|{r.ProcessGuid}|{r.QueryName}|{r.event_time}".encode()
#     ).hexdigest()[:16], axis=1),
#     rule_name="Office Application DNS Query to Non-Microsoft Domain",
#     mitre_technique="T1566.001",
#     severity="High",
#     status="new",
#     indicator=lambda d: d["QueryName"],
#     indicator_type="dns_domain")
# -----------------------------------------------------------------------------

-RECORD 0------------------------------------------------------------------------------------------------------
 alert_id        | f076845b1f6f3592                                                                            
 detected_at     | 2023-01-27 11:30:14.523                                                                     
 rule_name       | Office Application DNS Query to Non-Microsoft Domain                                        
 rule_version    | 1.0                                                                                         
 mitre_technique | T1566.001                                                                                   
 mitre_name      | Spearphishing Attachment                                                                    
 severity        | High                                                                                        
 status          | new                                                                                  

### Writing to the alert table

The brief asks for the result to be written to a fictitious alert table. In a
real deployment this would be a Delta or Iceberg table that the SOC queue reads
from, appended to rather than overwritten, and partitioned by detection date.

Parquet is used here so the notebook is self-contained.

In [22]:
# Write the alert to a fictitious alert table.
# In production: append to a partitioned Delta/Iceberg table the SOC reads from.
(alert_records.write
    .mode("overwrite")
    .format("parquet")
    .save("output/alerts"))

# Read it back to confirm the round trip.
spark.read.parquet("output/alerts").show(truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------
 alert_id        | f076845b1f6f3592                                                                            
 detected_at     | 2023-01-27 11:30:14.523                                                                     
 rule_name       | Office Application DNS Query to Non-Microsoft Domain                                        
 rule_version    | 1.0                                                                                         
 mitre_technique | T1566.001                                                                                   
 mitre_name      | Spearphishing Attachment                                                                    
 severity        | High                                                                                        
 status          | new                                                                                  

### Part 3.3: Enrichment

I enrich the indicator against **URLhaus (abuse.ch)**, chosen deliberately over
VirusTotal or OTX because it needs no API key. A reviewer can run this notebook
without credential setup, and there is no key to accidentally commit to the
repository. The call is wrapped so the notebook still executes end to end
without network access.

**The interpretation matters more than the lookup.**

`www.mediafire.com` is a legitimate file-sharing service. Every reputation feed
will correctly return *not malicious*. A detection that depended on a
threat-intelligence verdict would have missed this entirely.

The signal here is **categorical, not reputational**. The finding is not "this
domain is known-bad" - it is "a document editor has no business resolving a
file-sharing host." Adversaries deliberately stage payloads on legitimate
services precisely because those services pass reputation checks.

That distinction is, I think, the most important thing in this submission. A lot
of noisy or ineffective detections exist because a reputation score was
substituted for behavioural reasoning about what a process should be doing.

**In production I would enrich with three things rather than one:**

- **Category** - file host, paste site, dynamic DNS, URL shortener
- **Prevalence** - has this domain been seen in *our* environment before?
  First-seen-in-environment is frequently a stronger signal than any external
  verdict
- **Popularity rank** - position against Tranco or Umbrella top-1M, to separate
  "obscure domain nobody visits" from "extremely common site"

Reputation feeds are a fourth input, not the answer.

In [23]:
DOMAIN = alert_records.select("indicator").first()[0]

# Wrapped so the notebook runs end to end without network access.
try:
    resp = requests.post("https://urlhaus-api.abuse.ch/v1/host/",
                         data={"host": DOMAIN},
                         timeout=10)
    ti_status = resp.json().get("query_status", "unknown")
except Exception as exc:
    ti_status = f"offline ({exc})"

print(f"URLhaus lookup for {DOMAIN}: {ti_status}")
print("A clean result is the expected outcome here - see the note above on")
print("categorical vs reputational signal.")

intel_lookup = spark.createDataFrame(
    [(DOMAIN,
      "file-sharing",
      ti_status,
      "Legitimate service commonly abused for second-stage payload staging",
      "T1105")],
    ["indicator", "ti_category", "ti_status", "ti_context", "ti_technique"])

alerts_enriched = alert_records.join(intel_lookup, on="indicator", how="left")
alerts_enriched.show(truncate=False, vertical=True)

URLhaus lookup for www.mediafire.com: unknown
A clean result is the expected outcome here - see the note above on
categorical vs reputational signal.
-RECORD 0------------------------------------------------------------------------------------------------------
 indicator       | www.mediafire.com                                                                           
 alert_id        | f076845b1f6f3592                                                                            
 detected_at     | 2023-01-27 11:30:14.523                                                                     
 rule_name       | Office Application DNS Query to Non-Microsoft Domain                                        
 rule_version    | 1.0                                                                                         
 mitre_technique | T1566.001                                                                                   
 mitre_name      | Spearphishing Attachment                       

In [24]:
# Rendered as pandas purely for readability of the final single-row output.
# All processing above is PySpark; this is a display convenience only.
alerts_enriched.toPandas().T

,0
indicator,www.mediafire.com
alert_id,f076845b1f6f3592
detected_at,2023-01-27 11:30:14.523000
rule_name,Office Application DNS Query to Non-Microsoft ...
rule_version,1.0
mitre_technique,T1566.001
mitre_name,Spearphishing Attachment
severity,High
status,new
host,win-host-ctus-attack-range-212


## Summary

### Finding

One event matched the hypothesis:

**Microsoft Word (`WINWORD.EXE`) resolved `www.mediafire.com`.**

Word has no legitimate reason to resolve a consumer file-sharing host. This is
consistent with macro or embedded-object execution inside a delivered document
reaching out to stage a second-stage payload - T1566.001 followed by T1105.

### Approach

**Loading.** The Cribl-forwarded JSON lines promote a few fields to the top
level and leave the full Sysmon event as a JSON string in `_raw`.

**Parsing.** I inspected `_raw` before choosing a parse strategy and confirmed
it is flat JSON rather than XML or key=value. I parsed it with an explicit
schema rather than inference, so the parse is deterministic and will not shift
if the input sample changes.

**Filtering.** I profiled the `EventCode` distribution before writing any
detection. The dataset is dominated by file deletes and process access; DNS
queries are a small fraction. That subset was small enough to review manually,
which I did. The detection is deliberately simple because the reasoning, not
query complexity, is what proves the hypothesis.

### Detection logic and tuning

Two conditions: the querying process is an Office binary, and the queried domain
is not Microsoft telemetry.

Every legitimate Word query in the data resolved under `office.com` or
`office.net`. I chose a **domain-suffix** allowlist over an enumerated hostname
list, because Microsoft adds and retires CDN hosts frequently and an explicit
list goes stale and begins producing false positives without anyone noticing. I
match on the process **basename** rather than the path tail, so the rule cannot
be evaded by a file merely ending in `winword.exe`.

### Normalization, alerting, enrichment

Normalized to OCSF DNS Activity (`class_uid` 4003) so Sysmon DNS events share a
schema with firewall and resolver telemetry.

The alert row carries identity, verdict, pivot handles, and MITRE mapping. The
`alert_id` is deterministic, so re-running the detection over an overlapping
window produces the same id rather than duplicate queue entries.

On enrichment: MediaFire is legitimate and returns clean from reputation feeds.
The signal is categorical, not reputational - a document editor resolving a file
host. A detection depending on a TI verdict would have missed this.

### Limitations and next steps

- **Primary false negative.** If the macro spawns a separate process
  (`powershell.exe`, `rundll32.exe`, `mshta.exe`) and that process performs the
  DNS query, `Image` is no longer an Office binary and this rule will not fire.
  The fix is to join to EventCode 1 (process create, present in this dataset)
  and evaluate parent-process ancestry rather than the querying image alone.
  **This is the change I would prioritise.**
- **Allowlist scope.** An attacker using a subdomain under an allowlisted suffix
  would be suppressed. Mitigate by also validating the resolved address against
  Microsoft's published IP ranges, so domain and destination must agree.
- **Allowlist location.** Inline here for readability. In production it belongs
  in a versioned reference table so tuning does not require a code change.
- **No baselining.** With a longer collection window I would score on per-host
  domain rarity and first-seen-in-environment rather than a static allowlist.
- **False positive rate.** Zero on this dataset, but the DNS subset here is far
  too small to characterise one honestly. I would backtest over historical data
  and run in shadow mode before enabling this in production.
